# Colab / Kaggle: Balanced Rationale Generation (QEvasion)

Runs on **Colab** (Drive) or **Kaggle** (/kaggle/working, /kaggle/input). Modes:
1. **`script_mulerouter`**: run your existing `generate_rationale_dataset.py` command with balanced sampling.
2. **`granite_8b_local`**: generate rationales directly with `ibm-granite/granite-3.2-8b-instruct` on Colab GPU.

- Colab: Drive mount; data/output under Drive.
- Kaggle: output under /kaggle/working; data from HF or add Input dataset.
- Resume via run-signature checkpoint validation.

Outputs:
- CSV with rationale columns (`initial_reasoning`, `initial_verdict`, `corrective_reasoning`, `final_verdict`, `verdict_match`)
- JSONL for SFT/KD
- checkpoint JSON for resume


In [ ]:
!pip -q install -U transformers datasets accelerate bitsandbytes peft openai pandas scikit-learn

In [ ]:

import os, subprocess, gc
from pathlib import Path

# =============================================================================
# Environment: Colab (Drive) vs Kaggle (/kaggle/working, /kaggle/input) vs local
# =============================================================================
IN_KAGGLE = os.path.exists('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_KAGGLE:
    KAGGLE_WORKING = Path('/kaggle/working')
    KAGGLE_INPUT = Path('/kaggle/input')
    DRIVE_BASE = KAGGLE_WORKING
    COLAB_BASE = KAGGLE_WORKING
    print('✅ Running on Kaggle. Output: /kaggle/working, input: /kaggle/input')
    print('   Enable GPU: Notebook Settings (right sidebar) > Accelerator > GPU')
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive')
    COLAB_BASE = Path('/content')
    print(f"✅ Running on Colab. Drive: {DRIVE_BASE}")
else:
    DRIVE_BASE = Path.home()
    COLAB_BASE = Path.cwd()
    print('ℹ️ Not Colab/Kaggle; using local paths')

# ----------------------------------------------------------------------------
# Repo setup (auto-clone when missing). Required for script/audit/training cells.
# ----------------------------------------------------------------------------
DEFAULT_REPO_URL = 'https://github.com/gigibot-tech/CLARITY-SemEval-2026.git'
REPO_URL = DEFAULT_REPO_URL
CLONE_REPO_IF_MISSING = True
PERSIST_REPO_ON_DRIVE = True  # Colab: clone under Drive; Kaggle: clone under /kaggle/working

if IN_KAGGLE:
    REPO_DIR = KAGGLE_WORKING / 'CLARITY-SemEval-2026'
elif IN_COLAB and PERSIST_REPO_ON_DRIVE:
    REPO_DIR = DRIVE_BASE / 'CLARITY-SemEval-2026'
else:
    REPO_DIR = COLAB_BASE / 'CLARITY-SemEval-2026' if (IN_COLAB or IN_KAGGLE) else Path('/Users/andrearachetta/Desktop/CLARITY-SemEval-2026')

REPO_AVAILABLE = REPO_DIR.exists()
if not REPO_AVAILABLE and CLONE_REPO_IF_MISSING and REPO_URL:
    print(f'📥 Cloning repo: {REPO_URL} -> {REPO_DIR}')
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
        REPO_AVAILABLE = True
    except Exception as e:
        print(f'⚠️ Repo clone failed: {e}')
        REPO_AVAILABLE = False

WORK_DIR = REPO_DIR if REPO_AVAILABLE else COLAB_BASE
os.chdir(WORK_DIR)
print('Working dir:', WORK_DIR)
print('Python:', subprocess.check_output(['python', '--version']).decode().strip())
print('REPO_AVAILABLE:', REPO_AVAILABLE)
if REPO_AVAILABLE:
    print('REPO_DIR:', REPO_DIR)
else:
    print('ℹ️ Continuing without repo; granite_8b_local + HF dataset mode still works.')

# Project folders: Colab/Kaggle use DRIVE_BASE (= Drive or /kaggle/working)
PROJECT_BASE = DRIVE_BASE / 'granite_clarity'
DATA_DIR = PROJECT_BASE / 'data'
OUT_ROOT = PROJECT_BASE / 'qevasion_rationale'
KAGGLE_INPUT_DATASET = None
if IN_KAGGLE and KAGGLE_INPUT.exists():
    input_dirs = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
    if input_dirs:
        KAGGLE_INPUT_DATASET = input_dirs[0]
        print('Kaggle input dataset:', KAGGLE_INPUT_DATASET)

for d in [PROJECT_BASE, DATA_DIR, OUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('OUT_ROOT:', OUT_ROOT)

candidates = sorted(
    [x for x in DATA_DIR.glob('*') if x.suffix.lower() in {'.json', '.jsonl', '.csv', '.parquet'}],
    key=lambda x: x.stat().st_mtime,
    reverse=True,
)
if IN_KAGGLE and KAGGLE_INPUT_DATASET is not None:
    candidates += [x for x in KAGGLE_INPUT_DATASET.glob('*') if x.is_file() and x.suffix.lower() in {'.json', '.jsonl', '.csv', '.parquet'}]
if candidates:
    print(f'Found {len(candidates)} candidate data file(s) (newest first):')
    for f in candidates[:10]:
        print(' -', f)
else:
    print('No local dataset files in DATA_DIR; HF loading mode is recommended (DATA_SOURCE_MODE="hf").')


In [ ]:

from pathlib import Path

# --------- CONFIG ---------
RUN_MODE = 'granite_8b_local'  # 'script_mulerouter' or 'granite_8b_local'
SPLIT = 'train'
PER_LABEL_CAP = 250
SEED = 42
RESUME = True
FORCE_RESUME_DIFFERENT_CONFIG = False
LOG_EVERY = 5

# Dataset source (aligned with Granite-style "auto + explicit path" workflow)
DATA_SOURCE_MODE = 'hf'  # 'hf' or 'json'
HF_DATASET_ID = 'ailsntua/QEvasion'
LOCAL_DATA_FILE = DATA_DIR / f'qevasion_{SPLIT}.jsonl'  # used when DATA_SOURCE_MODE='json'

RUN_TAG = f'{SPLIT}_cap{PER_LABEL_CAP}_seed{SEED}_{RUN_MODE}'
OUT_DIR = OUT_ROOT / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUT_DIR / 'qevasion_rationale_raw.csv'
OUTPUT_JSONL = OUT_DIR / 'RationaleTraining_raw.jsonl'
CHECKPOINT_FILE = OUT_DIR / 'rationale_generation_checkpoint.json'

# Kaggle: path to Input dataset JSONL for training (add dataset 'gigibot/rationale-semeval2026' as Input)
KAGGLE_TRAINING_JSONL = Path('/kaggle/input/rationale-semeval2026/RationaleTraining_raw.jsonl') if IN_KAGGLE else None

# Granite local mode
GRANITE_MODEL_ID = 'ibm-granite/granite-3.2-8b-instruct'
LOAD_IN_4BIT = True
TEMPERATURE = 0.3
MAX_NEW_TOKENS = 900
PAUSE_SECONDS = 0.1

print('RUN_MODE:', RUN_MODE)
print('SPLIT:', SPLIT)
print('DATA_SOURCE_MODE:', DATA_SOURCE_MODE)
print('HF_DATASET_ID:', HF_DATASET_ID)
print('LOCAL_DATA_FILE:', LOCAL_DATA_FILE)
print('OUTPUT_CSV:', OUTPUT_CSV)
print('OUTPUT_JSONL:', OUTPUT_JSONL)
print('CHECKPOINT_FILE:', CHECKPOINT_FILE)
if IN_KAGGLE:
    print('KAGGLE_TRAINING_JSONL:', KAGGLE_TRAINING_JSONL, '(exists)' if (KAGGLE_TRAINING_JSONL and KAGGLE_TRAINING_JSONL.exists()) else '(add dataset as Input)')


In [ ]:

# Option A: Run your existing script command (balanced sampling added to script).
import subprocess

if RUN_MODE == 'script_mulerouter':
    if not REPO_AVAILABLE:
        raise RuntimeError('RUN_MODE=script_mulerouter requires repo files. Enable auto-clone or set REPO_URL.')

    cmd = [
        'python', 'generate_rationale_dataset.py',
        '--split', SPLIT,
        '--per-label-cap', str(PER_LABEL_CAP),
        '--seed', str(SEED),
        '--output-csv', str(OUTPUT_CSV),
        '--output-jsonl', str(OUTPUT_JSONL),
        '--checkpoint-file', str(CHECKPOINT_FILE),
    ]
    if RESUME:
        cmd.append('--resume')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping script mode; RUN_MODE != script_mulerouter')


In [ ]:

# Option B: Local Granite 3.2-8B rationale generation with balanced sampling.
import csv
import hashlib
import json
import random
import re
import time
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Clear caches to free memory before loading model (reduces chance of OOM kill)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
print("Cleared GPU/MPS/CPU caches.", flush=True)

if RUN_MODE != 'granite_8b_local':
    print('Skipping granite_8b_local mode.')
else:
    def load_qevasion_split(split_name: str):
        if DATA_SOURCE_MODE == 'hf':
            print(f"Loading HF dataset: {HF_DATASET_ID} [{split_name}]")
            return load_dataset(HF_DATASET_ID, split=split_name)
        if DATA_SOURCE_MODE == 'json':
            if not LOCAL_DATA_FILE.exists():
                raise FileNotFoundError(f'LOCAL_DATA_FILE not found: {LOCAL_DATA_FILE}')
            print(f"Loading local JSON dataset: {LOCAL_DATA_FILE}")
            return load_dataset('json', data_files={split_name: str(LOCAL_DATA_FILE)}, split=split_name)
        raise ValueError(f'Unsupported DATA_SOURCE_MODE: {DATA_SOURCE_MODE}')

    print('Loading dataset...')
    ds = load_qevasion_split(SPLIT)

    buckets = {}
    for i in range(len(ds)):
        lbl = str(ds[i].get('clarity_label', '')).strip()
        if not lbl:
            continue
        buckets.setdefault(lbl, []).append(i)

    rng = random.Random(SEED)
    selected = []
    for lbl, idxs in sorted(buckets.items()):
        k = min(PER_LABEL_CAP, len(idxs)) if PER_LABEL_CAP > 0 else len(idxs)
        take = rng.sample(idxs, k)
        selected.extend(take)
        print(f"{lbl}: selected {k}/{len(idxs)}")
    rng.shuffle(selected)
    print('Total selected:', len(selected))

    run_signature_raw = {
        'run_mode': RUN_MODE,
        'split': SPLIT,
        'per_label_cap': PER_LABEL_CAP,
        'seed': SEED,
        'data_source_mode': DATA_SOURCE_MODE,
        'hf_dataset_id': HF_DATASET_ID,
        'local_data_file': str(LOCAL_DATA_FILE),
        'granite_model_id': GRANITE_MODEL_ID,
        'selected_len': len(selected),
    }
    run_signature = hashlib.sha1(json.dumps(run_signature_raw, sort_keys=True).encode('utf-8')).hexdigest()
    print('Run signature:', run_signature[:12])

    print('Loading model:', GRANITE_MODEL_ID)
    quant_cfg = None
    if LOAD_IN_4BIT and torch.cuda.is_available():
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    tokenizer = AutoTokenizer.from_pretrained(GRANITE_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        GRANITE_MODEL_ID,
        device_map='auto',
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        quantization_config=quant_cfg,
    )
    model.eval()

    def extract_reasoning_and_verdict(response_text: str):
        reasoning = ''
        verdict = ''
        think_match = re.search(r'<think>(.*?)</think>', response_text, re.DOTALL)
        if think_match:
            reasoning = think_match.group(1).strip()
        else:
            vm = re.search(r'Verdict:\s*([^\n]+)', response_text, re.IGNORECASE)
            reasoning = response_text[:vm.start()].strip() if vm else response_text.strip()

        vm = re.search(r'Verdict:\s*([^\n]+)', response_text, re.IGNORECASE)
        if vm:
            v = vm.group(1).strip().lower()
            if 'clear non-reply' in v or 'non-reply' in v:
                verdict = 'Clear Non-Reply'
            elif 'clear reply' in v or v == 'clear':
                verdict = 'Clear Reply'
            elif 'ambivalent' in v:
                verdict = 'Ambivalent'
            else:
                verdict = vm.group(1).strip()
        return reasoning, verdict

    def generate_text(prompt: str):
        messages = [{'role': 'user', 'content': prompt}]
        try:
            formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            formatted = prompt
        inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    def initial_prompt(question, answer, clarity_label):
        return f'''The following interview question and answer have been labeled as: {clarity_label}.

Question: {question}
Answer: {answer}

Analyze the logic behind this label. Provide step-by-step reasoning, then conclude with a verdict.

Format your response as:
<think>
[Your reasoning here]
</think>
Verdict: [One of: "Clear Reply", "Clear Non-Reply", or "Ambivalent"]'''

    def corrective_prompt(question, answer, true_label, evasion_label, initial_verdict, initial_reasoning):
        evasion_context = f' The evasion type is: {evasion_label}.' if evasion_label else ''
        return f'''Actually, the correct label is {true_label}.{evasion_context}

Your initial analysis concluded: {initial_verdict}
Your initial reasoning: {initial_reasoning[:700]}...

Re-analyze where the mismatch happened and provide corrected reasoning.
Format your response as:
<think>
[Your corrected reasoning here]
</think>
Verdict: [Correct semantic label: "Clear Reply", "Clear Non-Reply", or "Ambivalent"]'''

    # Resume support with run-signature check
    start_pos = 0
    processed = 0
    matched = 0

    if RESUME and CHECKPOINT_FILE.exists():
        ck = json.loads(CHECKPOINT_FILE.read_text())
        old_sig = ck.get('run_signature', '')
        if old_sig and old_sig != run_signature and not FORCE_RESUME_DIFFERENT_CONFIG:
            raise ValueError(
                'Checkpoint exists but run signature changed. '
                'Set FORCE_RESUME_DIFFERENT_CONFIG=True to override, or clear checkpoint/output files.'
            )
        start_pos = int(ck.get('last_position', 0))
        processed = int(ck.get('processed', 0))
        matched = int(ck.get('matched', 0))
        print('Resuming from position:', start_pos)
    elif RESUME:
        print('No checkpoint found. Starting new run.')

    fieldnames = [
        'source_idx', 'title', 'date', 'president', 'url', 'question_order',
        'interview_question', 'interview_answer', 'question', 'index',
        'clarity_label', 'evasion_label',
        'initial_reasoning', 'initial_verdict', 'corrective_reasoning',
        'final_verdict', 'correction_applied', 'verdict_match'
    ]

    csv_exists = OUTPUT_CSV.exists()
    if not csv_exists:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    label_done = Counter()

    for pos in range(start_pos, len(selected)):
        src_i = selected[pos]
        ex = ds[src_i]

        q = str(ex.get('interview_question', ex.get('question', '')))
        a = str(ex.get('interview_answer', ''))
        y = str(ex.get('clarity_label', ''))
        ev = str(ex.get('evasion_label', '')) if ex.get('evasion_label') is not None else ''

        if not q or not a or not y:
            continue

        try:
            text_1 = generate_text(initial_prompt(q, a, y))
            init_reason, init_verdict = extract_reasoning_and_verdict(text_1)

            corr_reason = ''
            final_verdict = init_verdict
            corrected = False

            if init_verdict != y and init_verdict:
                corrected = True
                text_2 = generate_text(corrective_prompt(q, a, y, ev, init_verdict, init_reason))
                corr_reason, corrected_verdict = extract_reasoning_and_verdict(text_2)
                if corrected_verdict:
                    final_verdict = corrected_verdict

            verdict_match = (final_verdict == y)
            if verdict_match:
                matched += 1

            row = {
                'source_idx': src_i,
                'title': ex.get('title', ''),
                'date': ex.get('date', ''),
                'president': ex.get('president', ''),
                'url': ex.get('url', ''),
                'question_order': ex.get('question_order', ''),
                'interview_question': q,
                'interview_answer': a,
                'question': ex.get('question', ''),
                'index': ex.get('index', ''),
                'clarity_label': y,
                'evasion_label': ev,
                'initial_reasoning': init_reason,
                'initial_verdict': init_verdict,
                'corrective_reasoning': corr_reason,
                'final_verdict': final_verdict,
                'correction_applied': str(corrected),
                'verdict_match': str(verdict_match),
            }

            with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(row)

            if verdict_match:
                kd_record = {
                    'instruction': 'Analyze this politician answer for clarity/evasion.',
                    'input': f"Q: {q}\nA: {a}",
                    'output': f"<think>\n{init_reason}\n</think>\nVerdict: {final_verdict}",
                }
                with OUTPUT_JSONL.open('a', encoding='utf-8') as f:
                    f.write(json.dumps(kd_record, ensure_ascii=False) + '\n')

            processed += 1
            label_done[y] += 1

            if (processed % LOG_EVERY) == 0:
                ck_payload = {
                    'last_position': pos + 1,
                    'processed': processed,
                    'matched': matched,
                    'run_signature': run_signature,
                    'run_signature_raw': run_signature_raw,
                }
                CHECKPOINT_FILE.write_text(json.dumps(ck_payload, ensure_ascii=False, indent=2))
                print(
                    f"[{processed}] pos={pos+1}/{len(selected)} matched={matched} "
                    f"({(matched/max(1, processed))*100:.1f}%) labels_done={dict(label_done)}"
                )

            time.sleep(PAUSE_SECONDS)

        except Exception as e:
            print(f'Error at pos={pos}, src_idx={src_i}: {e}')

    final_ck = {
        'last_position': len(selected),
        'processed': processed,
        'matched': matched,
        'run_signature': run_signature,
        'run_signature_raw': run_signature_raw,
    }
    CHECKPOINT_FILE.write_text(json.dumps(final_ck, ensure_ascii=False, indent=2))

    print('Done.')
    print('Processed:', processed)
    print('Matched:', matched)
    print('Match rate:', (matched / max(1, processed)))
    print('Per-label processed:', dict(label_done))
    print('CSV:', OUTPUT_CSV)
    print('JSONL:', OUTPUT_JSONL)


In [ ]:

# Optional post-audit of the generated CSV (with streaming logs)
import subprocess

if not REPO_AVAILABLE:
    print('Skipping audit: repo not available (audit script lives in repo/scripts).')
else:
    AUDIT_DIR = OUT_DIR / 'rationale_dataset_audit'
    cmd = [
        'python', str(REPO_DIR / 'scripts' / 'audit_rationale_dataset.py'),
        '--input-csv', str(OUTPUT_CSV),
        '--output-dir', str(AUDIT_DIR),
        '--max-length', '1536',
    ]
    print('Running:', ' '.join(cmd))
    print('-' * 60)
    
    # Stream output line by line for live logging
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    
    print('-' * 60)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    print('✅ Audit complete!')
    print('Audit summary:', AUDIT_DIR / 'summary.json')


### Training on Kaggle (skip rationale generation)
1. Add dataset **gigibot/rationale-semeval2026** as Input (Notebook → Add Input → Datasets).
2. Run **Setup** and **Config** cells above.
3. Run the cell below (build CSV from JSONL), then run the **Training** cell.
4. The script `train_granite_rationale.py` comes from the cloned repo (no drag-and-drop).


In [ ]:

# Build a training CSV from JSONL so we can run train_granite_rationale.py.
# On Kaggle: use KAGGLE_TRAINING_JSONL (Input dataset). Else: use OUTPUT_JSONL from this run.
import csv
import json
import re
from collections import Counter

source_jsonl = (KAGGLE_TRAINING_JSONL if (IN_KAGGLE and KAGGLE_TRAINING_JSONL is not None and KAGGLE_TRAINING_JSONL.exists()) else OUTPUT_JSONL)
if not source_jsonl.exists():
    raise FileNotFoundError(f'JSONL not found: {source_jsonl}. On Kaggle, add dataset gigibot/rationale-semeval2026 as Input.')
TRAIN_FROM_JSONL_CSV = OUT_DIR / 'rationale_from_jsonl_for_training.csv'

required_cols = [
    'interview_question', 'interview_answer', 'clarity_label',
    'verdict_match', 'initial_reasoning', 'initial_verdict',
    'corrective_reasoning', 'final_verdict', 'correction_applied',
]
extra_cols = [
    'evasion_label', 'title', 'date', 'president', 'url',
    'question_order', 'question', 'index'
]
fieldnames = required_cols + extra_cols

def parse_qa(text: str):
    m = re.match(r'Q:\s*(.*?)\nA:\s*(.*)', text or '', re.DOTALL)
    if not m:
        return '', ''
    return m.group(1).strip(), m.group(2).strip()

def parse_reasoning_and_verdict(text: str):
    reasoning = ''
    verdict = ''
    think = re.search(r'<think>(.*?)</think>', text or '', re.DOTALL | re.IGNORECASE)
    if think:
        reasoning = think.group(1).strip()
    v = re.search(r'Verdict:\s*([^\n]+)', text or '', re.IGNORECASE)
    if v:
        verdict = v.group(1).strip()
    return reasoning, verdict

rows = []
label_counter = Counter()

with source_jsonl.open('r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        q, a = parse_qa(rec.get('input', ''))
        reasoning, verdict = parse_reasoning_and_verdict(rec.get('output', ''))
        if not q or not a or not verdict:
            continue
        label_counter[verdict] += 1
        rows.append({
            'interview_question': q,
            'interview_answer': a,
            'clarity_label': verdict,
            'verdict_match': 'True',
            'initial_reasoning': reasoning,
            'initial_verdict': verdict,
            'corrective_reasoning': '',
            'final_verdict': verdict,
            'correction_applied': 'False',
            'evasion_label': '',
            'title': '',
            'date': '',
            'president': '',
            'url': '',
            'question_order': '',
            'question': q,
            'index': i,
        })

with TRAIN_FROM_JSONL_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('TRAIN_FROM_JSONL_CSV:', TRAIN_FROM_JSONL_CSV)
print('Rows:', len(rows))
print('Label distribution:', dict(label_counter))


In [ ]:

# Train a single Granite model (LoRA fine-tune), not an ensemble.
import subprocess
import torch

if not REPO_AVAILABLE:
    print('Skipping training: repo scripts are not available in this runtime.')
else:
    TRAIN_OUTPUT_DIR = OUT_DIR / 'granite_3p2_2b_lora_from_jsonl'
    cmd = [
        'python', str(REPO_DIR / 'train_granite_rationale.py'),
        '--rationale-csv', str(TRAIN_FROM_JSONL_CSV),
        '--output-dir', str(TRAIN_OUTPUT_DIR),
        '--epochs', '2',
        '--batch-size', '1',
        '--gradient-accumulation-steps', '8',
        '--max-length', '1024',
        '--balance-rationale', 'downsample',
        '--eval',
    ]
    if torch.cuda.is_available():
        cmd.append('--load-8bit')

    print('Running:', ' '.join(cmd))
    # Multi-GPU: default run uses device_map="auto" (model across GPUs). For data-parallel (e.g. dual T4), run in a terminal: accelerate launch --num_processes=2 ... train_granite_rationale.py ...
    print('Tip: If you get CUDA OOM, use TPU: Runtime > Change runtime type > TPU, then run: !pip install torch_xla')
    # Stream stdout so you see training logs in the cell (subprocess.run would buffer until exit)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    print('Training output dir:', TRAIN_OUTPUT_DIR)
    print('Check output dir for: training_data_stats.json, training_summary.json, checkpoint-*, and final model files.')


In [ ]:

# Optional: merge LoRA adapter into one standalone Granite checkpoint for inference/export.
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if 'TRAIN_OUTPUT_DIR' not in globals():
    print('Run the training cell first.')
else:
    base_model_id = 'ibm-granite/granite-3.2-2b-instruct'
    merged_dir = Path(TRAIN_OUTPUT_DIR) / 'merged_full_model'
    merged_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    peft_model = PeftModel.from_pretrained(base_model, str(TRAIN_OUTPUT_DIR))
    merged_model = peft_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    print('Merged model saved to:', merged_dir)


In [ ]:
# Push trained model to HuggingFace Hub
# Set HF_TOKEN in Kaggle Secrets or Colab Secrets, or login via `huggingface-cli login`

from huggingface_hub import HfApi, login
import os

# CONFIG - change these!
HF_REPO_ID = 'gigibot/granite-clarity-lora'  # your HuggingFace repo (e.g., 'username/model-name')
PUSH_MERGED = True   # Push merged full model (larger, standalone)
PUSH_ADAPTER = True  # Push LoRA adapter only (smaller, requires base model)

# Authenticate
hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if hf_token:
    login(token=hf_token)
    print('✅ Logged in to HuggingFace Hub')
else:
    print('⚠️ No HF_TOKEN found. Add it to Kaggle Secrets or run: huggingface-cli login')

if 'TRAIN_OUTPUT_DIR' not in globals():
    print('Run the training cell first.')
else:
    api = HfApi()
    
    # Push LoRA adapter
    if PUSH_ADAPTER:
        print(f'📤 Pushing LoRA adapter to {HF_REPO_ID}...')
        api.upload_folder(
            folder_path=str(TRAIN_OUTPUT_DIR),
            repo_id=HF_REPO_ID,
            repo_type='model',
            ignore_patterns=['*.bin', 'optimizer.pt', 'scheduler.pt', 'training_args.bin'],
        )
        print(f'✅ LoRA adapter pushed: https://huggingface.co/{HF_REPO_ID}')
    
    # Push merged model (if it exists)
    merged_dir = Path(TRAIN_OUTPUT_DIR) / 'merged_full_model'
    if PUSH_MERGED and merged_dir.exists():
        merged_repo = f'{HF_REPO_ID}-merged'
        print(f'📤 Pushing merged model to {merged_repo}...')
        api.upload_folder(
            folder_path=str(merged_dir),
            repo_id=merged_repo,
            repo_type='model',
        )
        print(f'✅ Merged model pushed: https://huggingface.co/{merged_repo}')
    elif PUSH_MERGED:
        print('ℹ️ No merged model found. Run the merge cell first if you want to push it.')

In [ ]:
# Download outputs as zip (for Kaggle) or save to Drive (for Colab)
import shutil
from pathlib import Path

# What to include in the archive
INCLUDE_MODEL = True      # Include trained model weights
INCLUDE_OUTPUTS = True    # Include CSV, JSONL, checkpoints
ARCHIVE_NAME = 'granite_clarity_training_outputs'

if 'TRAIN_OUTPUT_DIR' not in globals():
    print('Run the training cell first.')
else:
    archive_dir = Path('/kaggle/working' if IN_KAGGLE else '/tmp') / ARCHIVE_NAME
    archive_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy outputs
    if INCLUDE_OUTPUTS:
        for f in OUT_DIR.glob('*.csv'):
            shutil.copy(f, archive_dir)
        for f in OUT_DIR.glob('*.jsonl'):
            shutil.copy(f, archive_dir)
        for f in OUT_DIR.glob('*.json'):
            shutil.copy(f, archive_dir)
        print(f'✅ Copied CSV/JSONL/JSON outputs to {archive_dir}')
    
    # Copy model (adapter files only, not optimizer state)
    if INCLUDE_MODEL:
        model_out = archive_dir / 'model'
        model_out.mkdir(exist_ok=True)
        for f in Path(TRAIN_OUTPUT_DIR).glob('*'):
            if f.is_file() and f.suffix in {'.json', '.safetensors', '.bin', '.txt', '.md'}:
                if 'optimizer' not in f.name and 'scheduler' not in f.name:
                    shutil.copy(f, model_out)
        print(f'✅ Copied model files to {model_out}')
    
    # Create zip
    zip_path = shutil.make_archive(str(archive_dir), 'zip', archive_dir)
    print(f'📦 Archive created: {zip_path}')
    
    if IN_KAGGLE:
        print('💡 Download from Kaggle: Output tab (right sidebar) after notebook completes')
    elif IN_COLAB:
        from google.colab import files
        files.download(zip_path)
    else:
        print(f'📁 Archive at: {zip_path}')